In [3]:
import pandas as pd
import numpy as np

In [4]:
credit=pd.read_csv("tmdb_5000_credits.csv")
movie=pd.read_csv("tmdb_5000_movies.csv")

In [5]:
movie=movie.merge(credit,on='title')

In [6]:
df=movie[['genres','id','keywords','cast','crew','overview','title','release_date']]

In [7]:
df.duplicated().sum()

np.int64(0)

In [8]:
df.dropna(inplace=True)
df['title'].drop_duplicates(inplace=True)

In [9]:
import ast

def fetch_director(obj):
        data = ast.literal_eval(obj)
        for i in data:
            if i['job'] == 'Director':
                return i['name']

df.loc[:, 'director'] = df['crew'].apply(fetch_director)

In [10]:
import ast
def get_ids_as_string(x):
    return ",".join([str(i['name']) for i in ast.literal_eval(x)])

df.loc[:, 'genres_ids'] = df['genres'].apply(get_ids_as_string)
df.loc[:, 'keywords_ids'] = df['keywords'].apply(get_ids_as_string)

In [11]:
def get_top_five_cast_ids(obj):
        data = ast.literal_eval(obj)
        return ",".join([str(i['name']) for i in data[:5]])

df.loc[:, 'cast_ids'] = df['cast'].apply(get_top_five_cast_ids)

In [12]:
df.drop(columns=["genres","keywords","cast","crew"],inplace=True)

In [13]:
df.dropna(inplace=True)

In [14]:
df.info()

<class 'pandas.DataFrame'>
Index: 4776 entries, 0 to 4808
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   id            4776 non-null   int64
 1   overview      4776 non-null   str  
 2   title         4776 non-null   str  
 3   release_date  4776 non-null   str  
 4   director      4776 non-null   str  
 5   genres_ids    4776 non-null   str  
 6   keywords_ids  4776 non-null   str  
 7   cast_ids      4776 non-null   str  
dtypes: int64(1), str(7)
memory usage: 2.7 MB


In [15]:
df['release_date'] = pd.to_datetime(df['release_date'])
df['release_year'] = df['release_date'].dt.year

In [16]:
df['overview_str'] = df['overview'].apply(lambda x: " ".join(x) if isinstance(x, list) else str(x))
df['overview'] = (df['overview_str'] + " " + 
              df['genres_ids'] + " " + 
              df['keywords_ids'] + " " + 
              df['cast_ids'] + " " + 
              df['director'])

df.drop(columns=["genres_ids","keywords_ids","release_date","cast_ids","director","overview_str"],inplace=True)

In [17]:
df['overview'][1]

"Captain Barbossa, long believed to be dead, has come back to life and is headed to the edge of the Earth with Will Turner and Elizabeth Swann. But nothing is quite as it seems. Adventure,Fantasy,Action ocean,drug abuse,exotic island,east india trading company,love of one's life,traitor,shipwreck,strong woman,ship,alliance,calypso,afterlife,fighter,pirate,swashbuckler,aftercreditsstinger Johnny Depp,Orlando Bloom,Keira Knightley,Stellan Skarsgård,Chow Yun-fat Gore Verbinski"

In [18]:
df['overview']=df['overview'].str.lower()

In [19]:
import string 
ex=string.punctuation
def remove_punch(test):
    return test.translate(str.maketrans("","",ex))
df['overview']=df['overview'].apply(remove_punch)

In [20]:
from nltk.corpus import stopwords
stop_words = set(stopwords.words('english'))

def remove_stopwords(text):
    return " ".join([word for word in text.split() if word not in stop_words])

df['overview'] = df['overview'].apply(remove_stopwords)

In [21]:
from nltk.stem import WordNetLemmatizer
lemmatizer = WordNetLemmatizer()
def lemmatize_words(text):
  return " ".join([lemmatizer.lemmatize(word) for word in text.split()])
df['overview']=df['overview'].apply(lemmatize_words)

In [22]:
from nltk import sent_tokenize,word_tokenize
import nltk
nltk.download('punkt_tab')
df['overview'] = df['overview'].apply(word_tokenize)

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [23]:
df['overview'] = df['overview'].apply(lambda x: " ".join(x) if isinstance(x, list) else x)

In [24]:
data=df

In [25]:
data.reset_index(drop=True, inplace=True)

In [26]:
from sklearn.feature_extraction.text import TfidfVectorizer

# TF-IDF is generally better at filtering out "noise" than CountVectorizer
tfidf = TfidfVectorizer(
    stop_words='english', 
    ngram_range=(1, 2), 
    min_df=3, 
    max_df=0.7 
)
vector_matrix = tfidf.fit_transform(data['overview'])

In [27]:
from sklearn.metrics.pairwise import cosine_similarity
similarity_matrix = cosine_similarity(vector_matrix)
print(f"Similarity matrix shape: {similarity_matrix.shape}")

Similarity matrix shape: (4776, 4776)


In [28]:
def recommend(movie):
    if movie not in data['title'].values:
        print("Movie not found.")
        return
    movie_index=data[data['title']==movie].index[0]
    movie_list=sorted(list(enumerate(similarity_matrix[movie_index])),reverse=True,key=lambda x:x[1])[1:6]
    for i in movie_list:
        print(data.iloc[i[0]].title)

In [37]:
recommend("Avatar")

Aliens
The Inhabited Island
Apollo 18
Supernova
The Matrix


In [ ]:
import pickle
with open("movies.pkl", "wb") as f:
    pickle.dump(data, f)

In [36]:
pickle.dump(similarity_matrix, open('similarity.pkl','wb'))